In [1]:
from ultralytics import YOLO

In [ ]:
path_yolo_folder = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\models\yolo"

os.makedirs(path_yolo_folder , exist_ok = True)

In [2]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [3]:
path_yolo = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\models\trained_models\flags_countours_yolov8x_v1\weights\best.pt"

model = YOLO(path_yolo) # model = YOLO('yolov12m.pt') this wrong no use v , D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\improve

In [4]:
path_yaml = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\improve\data.yaml"

model.train(
    data=path_yaml,
    epochs=20,
    imgsz=1024,
    batch=6,
    augment=True,
    degrees=30,            # زوايا دوران ±30°
    translate=0.2,         # إزاحة افقية/رأسية
    scale=0.2,             # تكبير/تصغير صغير
    shear=5.0,             # سحب (تشويه مائل)
    perspective=0.001,     # محاكاة تغيّر منظور الخ (للتشويه perspective)
    device=0, 
    verbose=True,
    patience=4,  # ← دي اللي بتفعل الـ Early Stopping
    project=r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\models\trained_models",
    name="mahamed",
)



Ultralytics 8.3.165  Python-3.9.21 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\improve\data.yaml, degrees=30, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\github lite\cv_projects\Identifying and locating flags from aerial photography\models\trained_models\flags_countours_yol

train: Scanning D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\improve\labels\train.cache... 72 images, 0 backgrounds, 0 corrupt: 100%|██████████| 72/72 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.10.0 ms, read: 1953.5972.0 MB/s, size: 506.9 KB)


val: Scanning D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\improve\labels\val.cache... 18 images, 1 backgrounds, 0 corrupt: 100%|██████████| 19/19 [00:00<?, ?it/s]


Plotting labels to D:\github lite\cv_projects\Identifying and locating flags from aerial photography\models\trained_models\mahamed2\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000313, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.000515625), 103 bias(decay=0.0)
Image sizes 1024 train, 1024 val
Using 8 dataloader workers
Logging results to D:\github lite\cv_projects\Identifying and locating flags from aerial photography\models\trained_models\mahamed2
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

       2/20      8.46G      4.057       79.8      2.275         12       1024: 100%|██████████| 12/12 [02:19<00:00, 11.66s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:05<00:00,  2.53s/it]

                   all         19         18      0.462     0.0278      0.034    0.00696



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       3/20      8.57G      4.358      49.61      2.376         11       1024:  25%|██▌       | 3/12 [00:45<02:15, 15.08s/it]


KeyboardInterrupt: 

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
from geopy.distance import geodesic

# ========== مصفوفة الكاميرا ==========
K = np.array([
    [2900, 0, 2048],
    [0, 2900, 1080],
    [0, 0, 1]
], dtype=np.float32)

dist = np.zeros((5,), dtype=np.float32)

# ========== تحميل نموذج YOLO ==========
yolo_path = r"D:\Saher Hassaballah\Downloads\best.pt"
model = YOLO(yolo_path)

# ========== تحميل صورة واحدة ==========
img_path = r"C:\Users\saher\Pictures\Screenshots\Screenshot 2025-07-02 012456.png"
frame = cv2.imread(img_path)

# بيانات الطائرة
drone_lat = 30.1
drone_lon = 30.6
drone_alt = 70  # بالأمتار

# ========== تشغيل النموذج ==========
results = model.predict(frame, conf=0.05)

for result in results:
    boxes = result.boxes.xyxy.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy()

    for box, cls_id in zip(boxes, classes):
        x1, y1, x2, y2 = map(int, box)
        label = model.names[int(cls_id)]

        # مركز البوكس
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        # تحويل لنقطة أرضية
        pixel = np.array([[cx, cy]], dtype=np.float32)
        norm = cv2.undistortPoints(np.expand_dims(pixel, axis=1), K, dist)
        ray = np.array([norm[0][0][0], norm[0][0][1], 1.0]).reshape(3, 1)

        scale = drone_alt / ray[2][0]
        ground_vector = ray * scale

        delta_north = ground_vector[1][0]
        delta_east = ground_vector[0][0]

        new_point = geodesic(meters=delta_north).destination((drone_lat, drone_lon), 0)
        new_point = geodesic(meters=delta_east).destination(new_point, 90)

        flag_lat = new_point.latitude
        flag_lon = new_point.longitude

        # رسم
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{label} ({flag_lat:.5f}, {flag_lon:.5f})", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)






# عرض الصورة
cv2.imshow("Flag Detection", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()


KeyboardInterrupt: 